# Feature Engineering
Create lags, moving averages, volatility, and momentum features

In [1]:
import pandas as pd
import numpy as np
import os
import sys

sys.path.insert(0, os.path.abspath('..'))
PROCESSED_DATA_DIR = '../data/processed'

print('Starting feature engineering')

Starting feature engineering


## Load Preprocessed Data

In [2]:
df = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'inflation_master.csv'))
df['Date'] = pd.to_datetime(df['Date'])
print(f'Loaded data: {df.shape}')
print(df.head())

Loaded data: (22, 8)
   Inflation_Rate       Date  Policy_Rate  Current_Account_Percent_GDP  \
0             4.6 2005-01-01          8.5                    -2.663822   
1             4.6 2006-01-01          8.5                    -5.299428   
2             4.6 2007-01-01          8.5                    -4.330416   
3             4.6 2008-01-01          8.5                    -9.543195   
4             4.6 2009-01-01          8.5                    -0.510386   

   GDP_Growth  Government_Debt_Percent_GDP  Government_Spending_Percent_GDP  \
0    6.241748                    90.604913                        20.195965   
1    7.668292                    88.699484                        21.147726   
2    6.796826                    84.994417                        20.046704   
3    5.950088                    84.994417                        19.211791   
4    3.538912                    86.063492                        20.958420   

   Trade_Balance  
0  -2.179493e+09  
1  -3.110581e+09  
2 

## Create Lag Features

In [3]:
lag_periods = [1, 3, 6, 12]
numeric_cols = df.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    for lag in lag_periods:
        df[f'{col}_lag{lag}'] = df[col].shift(lag)

print(f'After lags: {df.shape[1]} columns')

After lags: 36 columns


## Create Moving Average Features

In [4]:
ma_periods = [3, 6, 12]

for col in numeric_cols:
    for ma in ma_periods:
        df[f'{col}_ma{ma}'] = df[col].rolling(window=ma).mean()

print(f'After MA: {df.shape[1]} columns')

After MA: 57 columns


## Create Volatility Features

In [5]:
vol_periods = [6, 12]

for col in numeric_cols:
    for vol in vol_periods:
        df[f'{col}_vol{vol}'] = df[col].rolling(window=vol).std()

print(f'After volatility: {df.shape[1]} columns')

After volatility: 71 columns


## Create Momentum Features

In [6]:
momentum_periods = [1, 3, 12]

for col in numeric_cols:
    for mom in momentum_periods:
        df[f'{col}_pct_change{mom}'] = df[col].pct_change(periods=mom)

print(f'After momentum: {df.shape[1]} columns')

After momentum: 92 columns


## Create Target Variable

In [7]:
if 'Inflation_Rate' in df.columns:
    df['Inflation_Rate_target'] = df['Inflation_Rate'].shift(-1)
    print(f'Target missing: {df["Inflation_Rate_target"].isnull().sum()} / {len(df)}')

Target missing: 1 / 22


## Handle Missing Values and Save

In [8]:
df = df.ffill().bfill()

print(f'Final shape: {df.shape}')
print(f'Missing values: {df.isnull().sum().sum()}')

output_path = os.path.join(PROCESSED_DATA_DIR, 'inflation_model_ready.csv')
df.to_csv(output_path, index=False)
print(f'Saved to {output_path}')

feature_names = [col for col in df.columns if col not in ['Date', 'Inflation_Rate', 'Inflation_Rate_target']]
print(f'Created {len(feature_names)} features')
print('Dataset ready for modeling!')

Final shape: (22, 93)
Missing values: 0
Saved to ../data/processed\inflation_model_ready.csv
Created 90 features
Dataset ready for modeling!
